# 🤖 Notebook 2 — Model Building (Grade Prediction Version)
### Student Performance Prediction — Multi-Class Edition
**Owner:** Person 2

**Upload before running:**
`X_train_v2.csv`, `X_test_v2.csv`, `y_train_v2.csv`, `y_test_v2.csv`

**What changed from v1:**
- Target has 6 classes (A/B/C/D/E/F) instead of 2
- sklearn handles multi-class automatically — no extra code needed
- Metrics now use `average='weighted'` to account for class imbalance
- Added a Confusion Matrix per model in this notebook for a quick check

## Step 1 — Install & Import Libraries

In [ ]:
!pip install xgboost -q

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')
print('Libraries loaded ✅')

## Step 2 — Upload & Load Data

In [ ]:
from google.colab import files
print('Upload: X_train_v2.csv, X_test_v2.csv, y_train_v2.csv, y_test_v2.csv')
uploaded = files.upload()

In [ ]:
X_train = pd.read_csv('X_train_v2.csv')
X_test  = pd.read_csv('X_test_v2.csv')
y_train = pd.read_csv('y_train_v2.csv').squeeze()
y_test  = pd.read_csv('y_test_v2.csv').squeeze()

# Define grade order for consistent display throughout
grade_order = ['A', 'B', 'C', 'D', 'E', 'F']

# XGBoost requires numeric labels internally
# LabelEncoder converts A→0, B→1, C→2, D→3, E→4, F→5
# We keep the original string labels for display but use encoded for XGBoost
le = LabelEncoder()
le.fit(grade_order)   # fit in alphabetical/grade order for consistency
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f'Training: {X_train.shape[0]} students, {X_train.shape[1]} features')
print(f'Testing:  {X_test.shape[0]} students')
print(f'\nGrade classes: {grade_order}')
print(f'Training grade counts:\n{y_train.value_counts().reindex(grade_order)}')

## Step 3 — Model 1: Logistic Regression (Baseline)

In [ ]:
# Logistic Regression with multi_class='multinomial' handles 6 classes
# It learns a separate decision boundary for each grade vs all others
# solver='lbfgs' is the recommended optimizer for multinomial classification

print('Training Logistic Regression...')
lr = LogisticRegression(
    max_iter=2000,           # more iterations needed for 6-class problem
    multi_class='multinomial',
    solver='lbfgs',
    random_state=42
)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# weighted F1 accounts for the fact that some grades have more students than others
# (e.g. more C students than A students — weighted gives fair importance to all)
print('\n=== Logistic Regression ===')
print(f'Accuracy (weighted): {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'F1 Score (weighted): {f1_score(y_test, y_pred_lr, average="weighted"):.4f}')
print('\nPer-class breakdown:')
print(classification_report(y_test, y_pred_lr, labels=grade_order))

## Step 4 — Model 2: Random Forest + Tuning

In [ ]:
# Random Forest naturally handles multi-class — each tree votes for a grade
# and the majority grade wins
#
# GridSearchCV tries all combinations of these hyperparameters
# and picks the one with best cross-validation weighted F1

rf_params = {
    'n_estimators':      [100, 200, 300],
    'max_depth':         [None, 5, 10, 15],
    'min_samples_split': [2, 5]
}

print('Training Random Forest with GridSearchCV (~2 mins)...')
rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(
    rf, rf_params,
    cv=5,
    scoring='f1_weighted',   # optimize weighted F1 for multi-class
    n_jobs=-1,
    verbose=1
)
rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)

print(f'\nBest params: {rf_grid.best_params_}')
print('\n=== Random Forest ===')
print(f'Accuracy (weighted): {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'F1 Score (weighted): {f1_score(y_test, y_pred_rf, average="weighted"):.4f}')
print('\nPer-class breakdown:')
print(classification_report(y_test, y_pred_rf, labels=grade_order))

## Step 5 — Model 3: Gradient Boosting

In [ ]:
# Gradient Boosting builds trees sequentially, each correcting the previous
# For multi-class it trains one set of trees per class internally

print('Training Gradient Boosting...')
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

print('\n=== Gradient Boosting ===')
print(f'Accuracy (weighted): {accuracy_score(y_test, y_pred_gb):.4f}')
print(f'F1 Score (weighted): {f1_score(y_test, y_pred_gb, average="weighted"):.4f}')
print('\nPer-class breakdown:')
print(classification_report(y_test, y_pred_gb, labels=grade_order))

## Step 6 — Model 4: XGBoost + Tuning

In [ ]:
# XGBoost needs numeric labels — we use y_train_enc (A=0, B=1 ... F=5)
# objective='multi:softprob' tells XGBoost this is a multi-class problem
# and to output a probability for each class

xgb_params = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2]
}

print('Training XGBoost with GridSearchCV (~3 mins)...')
xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=6,
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False,
    verbosity=0
)
xgb_grid = GridSearchCV(
    xgb, xgb_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train_enc)   # train on numeric encoded labels

best_xgb = xgb_grid.best_estimator_

# Convert predictions back to letter grades for display
y_pred_xgb_enc = best_xgb.predict(X_test)
y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)

print(f'\nBest params: {xgb_grid.best_params_}')
print('\n=== XGBoost ===')
print(f'Accuracy (weighted): {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'F1 Score (weighted): {f1_score(y_test, y_pred_xgb, average="weighted"):.4f}')
print('\nPer-class breakdown:')
print(classification_report(y_test, y_pred_xgb, labels=grade_order))

## Step 7 — Results Summary Table

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_gb),
        accuracy_score(y_test, y_pred_xgb)
    ],
    'F1 Weighted': [
        f1_score(y_test, y_pred_lr,  average='weighted'),
        f1_score(y_test, y_pred_rf,  average='weighted'),
        f1_score(y_test, y_pred_gb,  average='weighted'),
        f1_score(y_test, y_pred_xgb, average='weighted')
    ]
})

results = results.sort_values('F1 Weighted', ascending=False).reset_index(drop=True)

print('='*50)
print('       MODEL COMPARISON (GRADE PREDICTION)')
print('='*50)
print(results.to_string(index=False))
print('='*50)
print(f'\n🏆 Best Model: {results.iloc[0]["Model"]}')

results.to_csv('results_summary_v2.csv', index=False)
print('Saved results_summary_v2.csv ✅')

## Step 8 — Save All Models

In [ ]:
# Save all 4 models + the label encoder (needed to decode XGBoost predictions)
# Person 3 and the web app (Notebook 4) need all of these

models_to_save = {
    'lr_v2':    lr,
    'rf_v2':    best_rf,
    'gb_v2':    gb,
    'xgb_v2':   best_xgb,
    'label_encoder_v2': le    # save encoder so v4 can decode XGBoost outputs
}

for name, obj in models_to_save.items():
    with open(f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f'Saved {name}.pkl ✅')

print('\nDownload ALL .pkl files + results_summary_v2.csv and share with Person 3')